# 05 — Evaluation, Interpretation & Personal Validation

This notebook covers all Day 4 work:
1. Comprehensive model comparison (Riegel, VDOT, Linear, RF, XGBoost/LGBM)
2. SHAP interpretability analysis for the best model
3. Personal Strava data validation (n=1)
4. Error analysis and model limitations

**Depends on:** Trained models from `04_model_training.ipynb`, processed data from `data/processed/`.

## Setup

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import shap
import joblib

from src.evaluate import regression_metrics, comparison_table
from src.baselines import riegel_predict, vdot_from_race, vdot_predict

sns.set_theme(style='whitegrid')
%matplotlib inline

/opt/anaconda3/envs/marathon/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load test data and trained models

In [ ]:
# Load train/test splits
train = pd.read_parquet('../data/processed/train.parquet')
test = pd.read_parquet('../data/processed/test.parquet')

# Define X and y for test set
# Adjust column names to match your feature engineering output
# feature_cols = [...]  
# X_test = test[feature_cols]
# y_test = test['marathon_time']

print(f'Train: {train.shape}, Test: {test.shape}')

In [ ]:
# Load trained models (saved from notebook 04)
# lr_model = joblib.load('../models/linear_regression.pkl')
# rf_model = joblib.load('../models/random_forest.pkl')
# gb_model = joblib.load('../models/gradient_boosting.pkl')

---

## Part 1: Comprehensive model comparison

Compare all models and baselines on the held-out test set using the same metrics: MAE, RMSE, MAPE, R².

### Generate predictions from all models

In [ ]:
# Baseline predictions (Riegel, VDOT) on test set
# riegel_preds = test.apply(
#     lambda row: riegel_predict(row['half_marathon_time'], 21.0975, 42.195), axis=1
# )

# ML model predictions
# lr_preds = lr_model.predict(X_test)
# rf_preds = rf_model.predict(X_test)
# gb_preds = gb_model.predict(X_test)

### Final comparison table

In [ ]:
# Build results dict and display comparison
# results = {
#     'Riegel': regression_metrics(y_test, riegel_preds),
#     'VDOT': regression_metrics(y_test, vdot_preds),
#     'Linear Regression': regression_metrics(y_test, lr_preds),
#     'Random Forest': regression_metrics(y_test, rf_preds),
#     'Gradient Boosting': regression_metrics(y_test, gb_preds),
# }
# comparison_table(results)

In [ ]:
# Improvement over Riegel: percentage reduction in MAE
# best_mae = results['Gradient Boosting']['MAE']  # or whichever is best
# riegel_mae = results['Riegel']['MAE']
# improvement = (riegel_mae - best_mae) / riegel_mae * 100
# print(f'Best model improves on Riegel by {improvement:.1f}% MAE reduction')

### Predicted vs actual scatter plot

In [ ]:
# Scatter plot with 1:1 line for best model
# fig, ax = plt.subplots(figsize=(8, 8))
# ax.scatter(y_test, gb_preds, alpha=0.6, edgecolors='k', linewidth=0.5)
# lims = [min(y_test.min(), gb_preds.min()) - 10, max(y_test.max(), gb_preds.max()) + 10]
# ax.plot(lims, lims, '--', color='red', label='Perfect prediction')
# ax.set_xlabel('Actual marathon time (min)')
# ax.set_ylabel('Predicted marathon time (min)')
# ax.set_title('Best Model: Predicted vs Actual')
# ax.legend()
# plt.tight_layout()
# plt.savefig('../results/figures/predicted_vs_actual.png', dpi=150)
# plt.show()

### Residual analysis

In [ ]:
# Residuals colored by speed tier
# residuals = y_test - gb_preds

# def speed_tier(time_min):
#     if time_min < 180: return 'Sub-3'
#     elif time_min < 240: return '3-4 hrs'
#     elif time_min < 300: return '4-5 hrs'
#     else: return '5+ hrs'

# tiers = y_test.apply(speed_tier)

# fig, ax = plt.subplots(figsize=(10, 6))
# for tier in ['Sub-3', '3-4 hrs', '4-5 hrs', '5+ hrs']:
#     mask = tiers == tier
#     ax.scatter(y_test[mask], residuals[mask], alpha=0.6, label=tier)
# ax.axhline(y=0, color='red', linestyle='--')
# ax.set_xlabel('Actual marathon time (min)')
# ax.set_ylabel('Residual (actual - predicted, min)')
# ax.set_title('Residuals by Speed Tier')
# ax.legend()
# plt.tight_layout()
# plt.savefig('../results/figures/residuals_by_tier.png', dpi=150)
# plt.show()

In [ ]:
# Error distribution — check for symmetry / systematic bias
# fig, ax = plt.subplots(figsize=(8, 5))
# ax.hist(residuals, bins=20, edgecolor='black', alpha=0.7)
# ax.axvline(x=0, color='red', linestyle='--')
# ax.set_xlabel('Residual (min)')
# ax.set_ylabel('Count')
# ax.set_title('Error Distribution')
# print(f'Mean residual: {residuals.mean():.2f} min')
# print(f'Median residual: {residuals.median():.2f} min')
# plt.tight_layout()
# plt.show()

### Performance by subgroup

In [ ]:
# Does the model work better for fast runners vs slow runners? Men vs women?
# Group by speed tier and compute MAE per group

# tier_metrics = pd.DataFrame({
#     'tier': tiers,
#     'abs_error': residuals.abs()
# }).groupby('tier')['abs_error'].agg(['mean', 'median', 'count'])
# tier_metrics.columns = ['MAE', 'Median AE', 'N']
# tier_metrics

---

## Part 2: SHAP interpretability

Use SHAP values to explain which features matter most and in which direction they push predictions.

In [ ]:
# Create SHAP explainer for the best model
# For tree-based models, use TreeExplainer (fast)
# explainer = shap.TreeExplainer(gb_model)
# shap_values = explainer.shap_values(X_test)

### Summary plot

Shows which features have the biggest impact overall and whether high/low values push predictions up or down.

In [ ]:
# shap.summary_plot(shap_values, X_test)
# plt.savefig('../results/figures/shap_summary.png', dpi=150, bbox_inches='tight')

### Dependence plots for top features

In [ ]:
# Dependence plot for the most important feature
# shap.dependence_plot(0, shap_values, X_test)
# plt.savefig('../results/figures/shap_dependence_1.png', dpi=150, bbox_inches='tight')

In [ ]:
# Dependence plot for the second most important feature
# shap.dependence_plot(1, shap_values, X_test)
# plt.savefig('../results/figures/shap_dependence_2.png', dpi=150, bbox_inches='tight')

### SHAP interpretation

*Write 1 paragraph interpreting the SHAP results. Which features drive predictions? Are there any surprises? Does the model's logic make intuitive sense given what you know about marathon training?*

---

## Part 3: Personal Strava validation

Test the model on your own training data as an n=1 case study. This is narrative, not statistical evidence.

In [ ]:
# Load your Strava export
# strava = pd.read_csv('../data/raw/personal_strava/activities.csv')
# strava.head()

In [ ]:
# Filter to the 16 weeks before your marathon
# marathon_date = pd.Timestamp('YYYY-MM-DD')  # fill in your race date
# training_start = marathon_date - pd.Timedelta(weeks=16)
# training_block = strava[
#     (strava['date'] >= training_start) & (strava['date'] < marathon_date)
# ]

In [ ]:
# Calculate the features your model expects
# avg_weekly_mileage = ...
# longest_run = ...
# avg_pace = ...
# half_marathon_time = ...  # from a race in that period, if you have one
# print(f'Avg weekly mileage: {avg_weekly_mileage:.1f}')
# print(f'Longest run: {longest_run:.1f}')

In [ ]:
# Build a single-row DataFrame matching your feature columns
# my_features = pd.DataFrame([{
#     # fill in your feature values here
# }])

# Run predictions
# my_model_pred = gb_model.predict(my_features)[0]
# my_riegel_pred = riegel_predict(half_marathon_time, 21.0975, 42.195)
# my_actual = ...  # your actual marathon finish time in minutes

# print(f'Model prediction:  {my_model_pred:.1f} min')
# print(f'Riegel prediction: {my_riegel_pred:.1f} min')
# print(f'Actual time:       {my_actual:.1f} min')
# print(f'Model error:       {my_model_pred - my_actual:.1f} min')
# print(f'Riegel error:      {my_riegel_pred - my_actual:.1f} min')

### Personal validation writeup

*"My model predicted X, Riegel predicted Y, I actually ran Z." Explain any discrepancy — race-day conditions, pacing decisions, fitness changes in the final weeks, etc. Note that this is n=1 and not statistically meaningful.*

---

## Part 4: Error analysis and model limitations

### Worst predictions

In [ ]:
# Identify the 10 worst predictions
# error_df = test.copy()
# error_df['predicted'] = gb_preds
# error_df['abs_error'] = (error_df['marathon_time'] - error_df['predicted']).abs()
# worst_10 = error_df.nlargest(10, 'abs_error')
# worst_10[['marathon_time', 'predicted', 'abs_error'] + feature_cols]

### Pattern in worst predictions

*Is there a pattern? Are the worst predictions all very fast runners? Very slow? Runners with missing/imputed data? Document what you find.*

### Known limitations

- **Self-reported data:** Vickers dataset is survey-based with no independent verification of race times or training volumes.
- **No race-day conditions:** The model has no weather, course difficulty, or altitude features.
- **No pacing strategy data:** How a runner executes the race (positive/negative split) is a major factor not captured here.
- **Small dataset:** ~493 usable runners limits the model's ability to learn complex patterns and generalize.
- **Selection bias:** Survey respondents are more engaged, experienced runners — the model may not generalize to casual first-time marathoners.
- **Missing half marathon data:** ~22% of half marathon times were imputed via Riegel formula, introducing a fixed-formula assumption into a portion of the training data.

### What would improve this with more time?

*Discuss concrete next steps:*
- *Link Afonseca training logs to scraped race results (probabilistic matching by demographics and predicted finish time)*
- *Add race-day weather features via Open-Meteo API*
- *Build Strava API integration to collect training data from consenting runners*
- *Use Bayesian or sequence modeling over training trajectories*
- *Conformal prediction or quantile regression for uncertainty quantification*

---

## Key findings

*Summarize the main takeaways from this notebook in 3-5 sentences. Which model won? By how much over Riegel? What does SHAP tell you? How did the personal validation go?*